# Phase 4 — Blocking Recall & Missed Match Analysis
### Amazon ML Challenge: Multi-Source Entity Resolution

**Goal:** Measure the recall of the candidate blocking rules against `train_ground_truth.tsv`, evaluate individual block contributions (ablation), and diagnose missed matches.

**Core Metrics:**
1. **Pair-Level Recall:**
   $$\text{Pair Recall} = \frac{\sum_{s_1} |\text{Candidates}(s_1) \cap \text{TrueMatches}(s_1)|}{\sum_{s_1} |\text{TrueMatches}(s_1)|}$$
2. **Entity-Level Coverage:** Percentage of Source 1 entities that captured at least one (or all) true matches.
3. **Block Ablation:** Marginal recall and redundancy contributed by each of the 8 blocks.
4. **Missed Match Diagnosis:** Deep-dive inspection into false negatives (why were they missed?).

In [2]:
from __future__ import annotations

import csv
import re
from collections import defaultdict, Counter
from pathlib import Path
from typing import Dict, List, Set, Tuple, Any

# Auto-detect project paths (Local and Colab compatible)
if Path("/content/AMAZON_ML_CHALLENGE-main").exists():
    ROOT = Path("/content/AMAZON_ML_CHALLENGE-main")
elif Path("/content/AMAZON_ML_CHALLENGE").exists():
    ROOT = Path("/content/AMAZON_ML_CHALLENGE")
elif Path("..").resolve().name == "AMAZON_ML_CHALLENGE":
    ROOT = Path("..").resolve()
else:
    ROOT = Path(".").resolve()

DATA = ROOT / "student_resource" / "dataset"
GT_FILE = DATA / "train" / "train_ground_truth.tsv"
CANDIDATES_FILE = ROOT / "data" / "processed" / "phase3" / "candidate_pairs.tsv"
REPORTS_DIR = ROOT / "reports" / "phase4"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {ROOT}")
print(f"Ground Truth File: {GT_FILE} (Exists: {GT_FILE.exists()})")
print(f"Candidate Pairs File: {CANDIDATES_FILE} (Exists: {CANDIDATES_FILE.exists()})")
print(f"Reports Directory: {REPORTS_DIR}")

## Step 1: Load Ground Truth Matches
Loads the ground truth file mapping each `source1_entity_id` to its true matched entity IDs in Source 2 and Source 3.

In [4]:
def load_ground_truth(gt_path: Path, max_rows: int = None) -> Dict[str, Set[str]]:
    """Loads train_ground_truth.tsv into a dictionary: s1_id -> set of matched_ids."""
    ground_truth = {}
    with open(gt_path, mode="r", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for i, row in enumerate(reader):
            s1_id = row["source1_entity_id"].strip()
            matches = {m.strip() for m in row["matched_entity_ids"].split(",") if m.strip()}
            ground_truth[s1_id] = matches
            if max_rows and i + 1 >= max_rows:
                break
    return ground_truth

# Demonstration loader (uncomment when ready):
# gt_map = load_ground_truth(GT_FILE, max_rows=50000)
# print(f"Loaded {len(gt_map):,} ground truth S1 records.")

## Step 2: Calculate Candidate Recall Metrics
Evaluates how many true matches were captured by the generated candidate pairs.

In [6]:
def evaluate_blocking_recall(
    candidate_path: Path,
    gt_map: Dict[str, Set[str]]
) -> Dict[str, Any]:
    """Calculates overall recall, entity coverage, and identifies missed pairs."""
    total_true_matches = sum(len(m) for m in gt_map.values())
    retrieved_true_matches = 0
    s1_with_full_recall = 0
    s1_with_partial_recall = 0
    s1_with_zero_recall = 0
    
    missed_matches = []
    
    with open(candidate_path, mode="r", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            s1_id = row["source1_entity_id"].strip()
            if s1_id not in gt_map:
                continue
                
            true_set = gt_map[s1_id]
            cands_set = {c.strip() for c in row["candidate_entity_ids"].split(",") if c.strip()}
            
            captured = true_set & cands_set
            missed = true_set - cands_set
            
            retrieved_true_matches += len(captured)
            
            if len(captured) == len(true_set):
                s1_with_full_recall += 1
            elif len(captured) > 0:
                s1_with_partial_recall += 1
            else:
                s1_with_zero_recall += 1
                
            for m in missed:
                missed_matches.append((s1_id, m))
                
    pair_recall = retrieved_true_matches / max(1, total_true_matches)
    entity_recall = (s1_with_full_recall + s1_with_partial_recall) / max(1, len(gt_map))
    
    results = {
        "total_ground_truth_pairs": total_true_matches,
        "retrieved_true_matches": retrieved_true_matches,
        "pair_recall": pair_recall,
        "entity_coverage": entity_recall,
        "s1_full_recall": s1_with_full_recall,
        "s1_partial_recall": s1_with_partial_recall,
        "s1_zero_recall": s1_with_zero_recall,
        "total_missed_pairs": len(missed_matches),
        "missed_samples": missed_matches[:20]
    }
    return results

## Step 3: Block Ablation Analysis
Measures the marginal recall and unique contribution of each of the 8 blocks to identify which blocks are essential and which are redundant.

In [8]:
def analyze_block_ablation(block_coverage_map: Dict[str, Set[Tuple[str, str]]], total_gt: int) -> Dict[str, Dict[str, float]]:
    """
    Calculates individual recall and marginal lift for each blocking rule.
    block_coverage_map: dict mapping block_name -> set of (s1_id, match_id) pairs captured.
    """
    ablation_stats = {}
    all_captured = set()
    for pairs in block_coverage_map.values():
        all_captured.update(pairs)
        
    total_captured = len(all_captured)
    
    for block_name, pairs in block_coverage_map.items():
        alone_recall = len(pairs) / max(1, total_gt)
        # Marginal loss: how many pairs are lost if this block is removed?
        without_block = set()
        for b, p in block_coverage_map.items():
            if b != block_name:
                without_block.update(p)
        unique_contribution = len(all_captured - without_block)
        unique_pct = unique_contribution / max(1, total_gt)
        
        ablation_stats[block_name] = {
            "individual_recall": alone_recall,
            "unique_matches": unique_contribution,
            "marginal_lift_pct": unique_pct
        }
        
    return ablation_stats

## Step 4: Missed Match Diagnosis (Failure Analysis)
Inspects why true matches failed all 8 blocking rules (e.g. drastic spelling divergence, missing fields, or different postal codes).

In [10]:
def diagnose_missed_matches(
    missed_pairs: List[Tuple[str, str]],
    s1_dict: Dict[str, Dict[str, str]],
    s2_dict: Dict[str, Dict[str, str]],
    s3_dict: Dict[str, Dict[str, str]],
    max_display: int = 10
):
    """Displays side-by-side attributes of missed true matches to diagnose failure modes."""
    print(f"=== DIAGNOSING {min(len(missed_pairs), max_display)} MISSED TRUE MATCHES ===\n")
    
    for idx, (s1_id, match_id) in enumerate(missed_pairs[:max_display], 1):
        rec1 = s1_dict.get(s1_id, {})
        rec2 = s2_dict.get(match_id) or s3_dict.get(match_id, {})
        source_label = "S2" if match_id.startswith("S2") else "S3"
        
        print(f"MISSED PAIR #{idx}: [Source 1 ({s1_id}) <--> {source_label} ({match_id})]")
        print(f"  S1 Name   : '{rec1.get('business_name_normalized', rec1.get('business_name', ''))}'")
        print(f"  {source_label} Name   : '{rec2.get('business_name_normalized', rec2.get('business_name', ''))}'")
        print(f"  S1 Address: '{rec1.get('business_address_normalized', rec1.get('business_address', ''))}'")
        print(f"  {source_label} Address: '{rec2.get('business_address_normalized', rec2.get('business_address', ''))}'")
        print(f"  S1 Country: '{rec1.get('country_normalized', rec1.get('country', ''))}' | {source_label} Country: '{rec2.get('country_normalized', rec2.get('country', ''))}'")
        print("-" * 100)

## Step 5: Execution & Recall Report Generation
Runs the evaluation once `candidate_pairs.tsv` is available and compiles findings into `reports/phase4/recall_report.md`.

In [12]:
# Execution pipeline ready for when candidate_pairs.tsv is provided
def run_full_recall_evaluation(candidate_file: Path, report_out: Path):
    if not candidate_file.exists():
        print(f"Candidate file not found at: {candidate_file}")
        print("Please provide or generate candidate_pairs.tsv first.")
        return
        
    print("Loading Ground Truth...")
    gt = load_ground_truth(GT_FILE)
    print(f"Evaluating {candidate_file.name} against {len(gt):,} Ground Truth entities...")
    results = evaluate_blocking_recall(candidate_file, gt)
    
    print(f"\n=== RECALL EVALUATION RESULTS ===")
    print(f"- Pair Recall        : {results['pair_recall']*100:.2f}%")
    print(f"- Entity Coverage    : {results['entity_coverage']*100:.2f}%")
    print(f"- Captured Matches   : {results['retrieved_true_matches']:,} / {results['total_ground_truth_pairs']:,}")
    print(f"- Missed Matches     : {results['total_missed_pairs']:,}")
    
    # Save report to markdown
    with open(report_out, "w", encoding="utf-8") as f:
        f.write("# Phase 4: Blocking Recall Evaluation Report\n\n")
        f.write(f"- **Candidate File**: `{candidate_file.name}`\n")
        f.write(f"- **Pair-Level Recall**: {results['pair_recall']*100:.2f}%\n")
        f.write(f"- **Entity Coverage**: {results['entity_coverage']*100:.2f}%\n")
        f.write(f"- **Captured Matches**: {results['retrieved_true_matches']:,} / {results['total_ground_truth_pairs']:,}\n")
        f.write(f"- **Missed Matches**: {results['total_missed_pairs']:,}\n")
    print(f"Saved recall evaluation report to: {report_out}")

# Call when candidate_pairs.tsv is ready:
# run_full_recall_evaluation(CANDIDATES_FILE, REPORTS_DIR / "recall_report.md")
print("Recall evaluation module ready. Waiting for candidate_pairs.tsv.")